# Cathey — Software-Only Semantic Test

This is a pure software test on Mac, without using any Pi hardware. It supports:

- **Text Input**: Enter sentences directly in the notebook

- **Microphone Input**: Record audio using your computer's microphone → Faster Whisper → Agent

## 1. Mock Hardware & Initialization

In [1]:
import sys, logging
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)

# ── Mock GPIO (replaces lgpio / Pi hardware) ──────────────────────────────────
class MockGPIOExecutor:
    """Simulates hardware execution. Prints state changes, no lgpio required."""

    def __init__(self):
        self._state = {
            'light': 'off', 'brightness': 2, 'color_temp': 3,
            'curtain_pos': 0, 'window_pos': 0,
            'ac': 'off', 'ac_temp': None,
        }

    def get_device_state(self) -> dict:
        return {
            'color_temp':  self._state['color_temp'],
            'brightness':  self._state['brightness'],
            'curtain_pos': self._state['curtain_pos'],
            'window_pos':  self._state['window_pos'],
        }

    def execute(self, cmd: dict) -> str:
        device, action, value = cmd.get('device'), cmd.get('action'), cmd.get('value')
        label = f'[MOCK HW] {device} → {action}' + (f'({value})' if value is not None else '')
        if device == 'light':
            if action == 'turn_on':         self._state['light'] = 'on'
            elif action == 'turn_off':      self._state['light'] = 'off'
            elif action == 'set_brightness':
                self._state['brightness'] = 1 if value <= 33 else (2 if value <= 66 else 3)
            elif action == 'set_color_temp': self._state['color_temp'] = value
        elif device == 'curtain':
            self._state['curtain_pos'] = {'open': 100, 'close': 0}.get(action, value or self._state['curtain_pos'])
        elif device == 'window':
            self._state['window_pos']  = {'open': 100, 'close': 0}.get(action, value or self._state['window_pos'])
        elif device == 'ac':
            if action == 'turn_on':    self._state['ac'] = 'on'
            elif action == 'turn_off': self._state['ac'] = 'off'
            elif action == 'set_temperature': self._state['ac_temp'] = value
        print(label)
        return label

    def cleanup(self): pass


# ── Mock TTS (replaces Piper / pw-play) ──────────────────────────────────────
def mock_speak(text: str, verbose: bool = True):
    print(f'[Cathey] {text}')


print('Mock hardware ready.')

Mock hardware ready.


In [2]:
from sentence_transformers import SentenceTransformer
from nlp.llm_parser import LLMParser
from core.memory import MemoryManager
from core.agent import CatheyAgent
from core.config import EMBED_MODEL_NAME

print('Loading LLM ...')
llm    = LLMParser()
embed  = SentenceTransformer(EMBED_MODEL_NAME)
memory = MemoryManager(embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist())
gpio   = MockGPIOExecutor()

agent = CatheyAgent(llm=llm, memory=memory, speak=mock_speak, gpio=gpio)
embed.encode('warmup', convert_to_numpy=True)
print('Agent ready.')

Loading LLM ...
Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM ready.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Agent ready.


## 2. Text Input Test

In [3]:
import json, time

def test(text: str, reset: bool = False):
    """Send one utterance to the agent and pretty-print the result."""
    if reset:
        agent.reset_dialogue()
    print(f'\n>>> {text}')
    t0 = time.perf_counter()
    result = agent.handle(text, verbose=False)
    elapsed = (time.perf_counter() - t0) * 1000
    sem = result.get('semantic', {})
    print(f'  type    : {sem.get("type")}')
    if sem.get('type') == 'direct_command':
        print(f'  device  : {sem.get("device")}  action: {sem.get("action")}  value: {sem.get("value")}')
    elif sem.get('type') == 'needs_clarification':
        print(f'  question: {sem.get("question")}')
        print(f'  options : {sem.get("options")}')
    elif sem.get('type') == 'general_qa':
        print(f'  answer  : {sem.get("answer")}')
    print(f'  reply   : {result.get("spoken_reply")}')
    print(f'  latency : {elapsed:.0f} ms  (llm: {result.get("llm_latency_ms", 0):.0f} ms)')
    print(f'  reason  : {result.get("reason")}')
    return result

In [4]:
# ── Direct commands (should hit rule-based fast path) ─────────────────────────
test('Cathey, turn on the light.')
test('Cathey, open the curtain.')
test('Cathey, set the AC to 24 degrees.')
test('Cathey, turn off the light.')


>>> Cathey, turn on the light.
[MOCK HW] light → turn_on
[Cathey] Sure, turning on the light.
  type    : direct_command
  device  : light  action: turn_on  value: None
  reply   : Sure, turning on the light.
  latency : 245 ms  (llm: 0 ms)
  reason  : ok

>>> Cathey, open the curtain.
[MOCK HW] curtain → open
[Cathey] Sure, opening the curtain.
  type    : direct_command
  device  : curtain  action: open  value: None
  reply   : Sure, opening the curtain.
  latency : 210 ms  (llm: 0 ms)
  reason  : ok

>>> Cathey, set the AC to 24 degrees.
[MOCK HW] ac → set_temperature(24)
[Cathey] Sure, setting AC to 24 degrees.
  type    : direct_command
  device  : ac  action: set_temperature  value: 24
  reply   : Sure, setting AC to 24 degrees.
  latency : 215 ms  (llm: 0 ms)
  reason  : ok

>>> Cathey, turn off the light.
[MOCK HW] light → turn_off
[Cathey] Sure, turning off the light.
  type    : direct_command
  device  : light  action: turn_off  value: None
  reply   : Sure, turning off the

{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'light',
  'action': 'turn_off',
  'value': None,
  'reply': 'Sure, turning off the light.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'LIGHT -> OFF',
 'spoken_reply': 'Sure, turning off the light.',
 'llm_latency_ms': 0.0}

In [5]:
# ── Feelings / ambiguous (should trigger needs_clarification) ─────────────────
test('Cathey, I feel cold.')
test('Cathey, it is too dark in here.')
test('Cathey, I am hot.')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



>>> Cathey, I feel cold.
[MOCK HW] window → close
[Cathey] Sure, window close. (based on your past preference)
  type    : needs_clarification
  question: Do you also want to lower the AC temperature?
  options : ['lower_ac_temperature']
  reply   : Sure, window close. (based on your past preference)
  latency : 5067 ms  (llm: 4978 ms)
  reason  : procedural_memory_auto_resolved

>>> Cathey, it is too dark in here.
[Cathey] Would you like me to turn on the light? Here are your options: Option 1: Turn on the light. Please say the option number or describe what you want.
  type    : needs_clarification
  question: Would you like me to turn on the light?
  options : ['turn_on_light']
  reply   : Would you like me to turn on the light? Here are your options: Option 1: Turn on the light. Please say the option number or describe what you want.
  latency : 3092 ms  (llm: 3075 ms)
  reason  : clarification_requested

>>> Cathey, I am hot.
[Cathey] Would you like me to open the window or adjus

{'prefilter_passed': True,
 'semantic': {'type': 'needs_clarification',
  'question': 'Would you like me to open the window or adjust the air conditioning?',
  'options': ['open_window', 'adjust_ac_temperature'],
  'reply': 'Would you like me to open the window or adjust the air conditioning?'},
 'valid': True,
 'reason': 'clarification_requested',
 'execution': 'PENDING_USER_REPLY',
 'spoken_reply': 'Would you like me to open the window or adjust the air conditioning? Here are your options: Option 1: Open the window. Option 2: Adjust ac temperature. Please say the option number or describe what you want.',
 'llm_latency_ms': 2955.861}

In [6]:
# ── General QA (unrelated to device control) ──────────────────────────────────
test('Cathey, what is your name?')
test('Cathey, what time is it?')
test('Cathey, tell me a joke.')


>>> Cathey, what is your name?
[Cathey] Sorry, I didn't understand that.
  type    : invalid
  reply   : Sorry, I didn't understand that.
  latency : 896 ms  (llm: 886 ms)
  reason  : invalid_semantic_result

>>> Cathey, what time is it?
[Cathey] I don't have real-time information about the current time. Please check a clock for the accurate time.
  type    : general_qa
  answer  : I don't have real-time information about the current time. Please check a clock for the accurate time.
  reply   : I don't have real-time information about the current time. Please check a clock for the accurate time.
  latency : 2808 ms  (llm: 2792 ms)
  reason  : general_qa_answered

>>> Cathey, tell me a joke.
[Cathey] Sorry, I didn't understand that.
  type    : invalid
  reply   : Sorry, I didn't understand that.
  latency : 870 ms  (llm: 862 ms)
  reason  : invalid_semantic_result


{'prefilter_passed': True,
 'semantic': {'type': 'invalid'},
 'valid': False,
 'reason': 'invalid_semantic_result',
 'execution': 'SKIPPED',
 'spoken_reply': "Sorry, I didn't understand that.",
 'llm_latency_ms': 861.546}

In [7]:
# ── Clarification flow (two-turn dialogue) ────────────────────────────────────
agent.reset_dialogue()
test('Cathey, it is stuffy in here.')   # → needs_clarification
test('open the window')                  # → follow-up, should resolve to direct_command


>>> Cathey, it is stuffy in here.
[Cathey] Would you like me to open a window or use air conditioning? Here are your options: Option 1: Open the window. Option 2: Adjust air conditioning. Please say the option number or describe what you want.
  type    : needs_clarification
  question: Would you like me to open a window or use air conditioning?
  options : ['open_window', 'adjust_air_conditioning']
  reply   : Would you like me to open a window or use air conditioning? Here are your options: Option 1: Open the window. Option 2: Adjust air conditioning. Please say the option number or describe what you want.
  latency : 3097 ms  (llm: 3073 ms)
  reason  : clarification_requested

>>> open the window
[Cathey] Opening the window.
  type    : direct_command
  device  : window  action: open  value: None
  reply   : Opening the window.
  latency : 2459 ms  (llm: 2438 ms)
  reason  : ok


{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'window',
  'action': 'open',
  'value': None,
  'reply': 'Opening the window.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'WINDOW -> OPEN',
 'spoken_reply': 'Opening the window.',
 'llm_latency_ms': 2437.961}

In [8]:
# ── LLM path: set brightness / color temp via natural language ────────────────
test('Cathey, set the brightness to 70 percent.')
test('Cathey, make the light warmer.')
test('Cathey, close the curtain halfway.')


>>> Cathey, set the brightness to 70 percent.
[MOCK HW] light → set_brightness(70)
[Cathey] Sure, setting brightness to 70 percent.
  type    : direct_command
  device  : light  action: set_brightness  value: 70
  reply   : Sure, setting brightness to 70 percent.
  latency : 7 ms  (llm: 0 ms)
  reason  : ok

>>> Cathey, make the light warmer.
[MOCK HW] light → set_color_temp(4)
[Cathey] Done.
  type    : direct_command
  device  : light  action: set_color_temp  value: 4
  reply   : Done.
  latency : 8 ms  (llm: 0 ms)
  reason  : ok

>>> Cathey, close the curtain halfway.
[MOCK HW] curtain → close
[Cathey] Sure, closing the curtain.
  type    : direct_command
  device  : curtain  action: close  value: None
  reply   : Sure, closing the curtain.
  latency : 7 ms  (llm: 0 ms)
  reason  : ok


{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'curtain',
  'action': 'close',
  'value': None,
  'reply': 'Sure, closing the curtain.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'CURTAIN -> CLOSE',
 'spoken_reply': 'Sure, closing the curtain.',
 'llm_latency_ms': 0.0}

In [9]:
# ── Batch test with accuracy tracking ─────────────────────────────────────────
TEST_CASES = [
    # (input_text,                                    expected_type,             expected_device, expected_action)
    ('Cathey, turn on the light.',                    'direct_command',          'light',         'turn_on'),
    ('Cathey, turn off the light.',                   'direct_command',          'light',         'turn_off'),
    ('Cathey, open the curtain.',                     'direct_command',          'curtain',       'open'),
    ('Cathey, close the window.',                     'direct_command',          'window',        'close'),
    ('Cathey, set the AC to 22 degrees.',             'direct_command',          'ac',            'set_temperature'),
    ('Cathey, set brightness to 50.',                 'direct_command',          'light',         'set_brightness'),
    ('Cathey, RGB mode please.',                      'direct_command',          'light',         'rgb_cycle'),
    ('Cathey, I feel cold.',                          'needs_clarification',     None,            None),
    ('Cathey, it is too dark.',                       'needs_clarification',     None,            None),
    ('Cathey, I am hot.',                             'needs_clarification',     None,            None),
    ('Cathey, it is stuffy.',                         'needs_clarification',     None,            None),
    ('Cathey, what is your name?',                    'general_qa',              None,            None),
    ('Cathey, what is the capital of France?',        'general_qa',              None,            None),
    ('Cathey, hello.',                                'general_qa',              None,            None),
]

correct = 0
rows = []
for text, exp_type, exp_device, exp_action in TEST_CASES:
    agent.reset_dialogue()
    r = agent.handle(text, verbose=False)
    sem = r.get('semantic', {})
    got_type   = sem.get('type')
    got_device = sem.get('device')
    got_action = sem.get('action')
    ok_type   = got_type == exp_type
    ok_device = (exp_device is None) or (got_device == exp_device)
    ok_action = (exp_action is None) or (got_action == exp_action)
    passed = ok_type and ok_device and ok_action
    correct += int(passed)
    rows.append((text[:50], exp_type, got_type, got_device, got_action, '✓' if passed else '✗'))

print(f'\nAccuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES)*100:.1f}%\n')
print(f'{"Input":<52} {"Expected":<22} {"Got":<22} {"Device":<10} {"Action":<18} OK')
print('-' * 130)
for row in rows:
    print(f'{row[0]:<52} {row[1]:<22} {row[2]:<22} {str(row[3]):<10} {str(row[4]):<18} {row[5]}')

[MOCK HW] light → turn_on[Cathey] Sure, turning on the light.

[MOCK HW] light → turn_off
[Cathey] Sure, turning off the light.
[MOCK HW] curtain → open
[Cathey] Sure, opening the curtain.
[MOCK HW] window → close
[Cathey] Sure, closing the window.
[MOCK HW] ac → set_temperature(22)
[Cathey] Sure, setting AC to 22 degrees.
[MOCK HW] light → set_brightness(50)
[Cathey] Sure, setting brightness to 50 percent.
[MOCK HW] light → rgb_cycle
[Cathey] Sure, starting RGB cycle.
[MOCK HW] window → close[Cathey] Sure, window close. (based on your past preference)

[Cathey] Would you like me to turn on the light or open the curtain? Here are your options: Option 1: Turn on the light. Option 2: Open the curtain. Please say the option number or describe what you want.
[Cathey] Would you like me to open the window or adjust the air conditioning? Here are your options: Option 1: Open the window. Option 2: Adjust ac temperature. Please say the option number or describe what you want.
[MOCK HW] window →

## 3. Microphone Input (Optional)

In [ ]:
# ── loading faster-whisper────────────────────────────────
import io, time, numpy as np, sounddevice as sd, soundfile as sf
from collections import deque
from faster_whisper import WhisperModel
from core.config import (
    SAMPLE_RATE, CHANNELS, AUDIO_DTYPE,
    ENERGY_THRESHOLD, FRAME_SAMPLES, FRAME_DURATION,
    SILENCE_FRAMES, PRE_ROLL_FRAMES, MAX_FRAMES, MIN_SPEECH_SECONDS,
)

print('Loading STT (tiny.en) ...')
_stt = WhisperModel('tiny.en', device='cpu', compute_type='int8')
print('STT ready.')

def _transcribe(audio: np.ndarray) -> str:
    buf = io.BytesIO()
    sf.write(buf, audio, SAMPLE_RATE, format='wav')
    buf.seek(0)
    segs, _ = _stt.transcribe(
        buf, beam_size=1, language='en',
        initial_prompt='Cathey, I feel cold. Turn on the light. Open the curtain.',
        condition_on_previous_text=False,
    )
    return ' '.join(s.text.strip() for s in segs).strip()

def _collect_utterance(stream):
    pre_buf = deque(maxlen=PRE_ROLL_FRAMES)
    collected, started, silence_count, speech_frames = [], False, 0, 0
    while True:
        frame, _ = stream.read(FRAME_SAMPLES)
        frame = frame.copy()
        energy = float(np.mean(np.abs(frame)))
        if not started:
            pre_buf.append(frame)
            if energy >= ENERGY_THRESHOLD:
                started = True
                collected.extend(list(pre_buf))
                collected.append(frame)
                speech_frames += 1
        else:
            collected.append(frame)
            if energy >= ENERGY_THRESHOLD:
                speech_frames += 1; silence_count = 0
            else:
                silence_count += 1
            if silence_count >= SILENCE_FRAMES or len(collected) >= MAX_FRAMES:
                break
    if not started or speech_frames * FRAME_DURATION < MIN_SPEECH_SECONDS:
        return None
    return np.concatenate(collected, axis=0)

print('Mic helper ready. Run the next cell to start listening.')

Loading STT (tiny.en) ...
STT ready.
Mic helper ready. Run the next cell to start listening.


In [ ]:
# ── Single microphone recording + agent processing ─────────────────────────────────────────────
# Speak after running, pause for 0.5 seconds after speaking, then automatically end.
print('Listening... speak now.')
with sd.InputStream(
    samplerate=SAMPLE_RATE, channels=CHANNELS,
    dtype=AUDIO_DTYPE, blocksize=FRAME_SAMPLES,
) as stream:
    time.sleep(0.1)
    audio = _collect_utterance(stream)

if audio is None:
    print('No speech detected.')
else:
    t0 = time.perf_counter()
    text = _transcribe(audio)
    stt_ms = (time.perf_counter() - t0) * 1000
    print(f'[STT {stt_ms:.0f} ms] {text!r}')
    test(text)

Listening... speak now.
[STT 516 ms] "Cathey, what's our name?"

>>> Cathey, what's our name?
[Cathey] Is your question about the assistant's name or the user's name? Here are your options: Option 1: Assistant name. Option 2: User name. Please say the option number or describe what you want.
  type    : needs_clarification
  question: Is your question about the assistant's name or the user's name?
  options : ['assistant_name', 'user_name']
  reply   : Is your question about the assistant's name or the user's name? Here are your options: Option 1: Assistant name. Option 2: User name. Please say the option number or describe what you want.
  latency : 4693 ms  (llm: 3434 ms)
  reason  : clarification_requested


In [ ]:
# ── Continuous microphone loop (Interrupt kernel to stop)───────────────────────────────────
print('Continuous mic mode. Press Interrupt (■) in the toolbar to stop.\n')
try:
    while True:
        print('[Listening] ...')
        with sd.InputStream(
            samplerate=SAMPLE_RATE, channels=CHANNELS,
            dtype=AUDIO_DTYPE, blocksize=FRAME_SAMPLES,
        ) as stream:
            time.sleep(0.1)
            audio = _collect_utterance(stream)
        if audio is None:
            continue
        t0 = time.perf_counter()
        text = _transcribe(audio)
        stt_ms = (time.perf_counter() - t0) * 1000
        print(f'[STT {stt_ms:.0f} ms] {text!r}')
        test(text)
except KeyboardInterrupt:
    print('\nStopped.')

Continuous mic mode. Press Interrupt (■) in the toolbar to stop.

[Listening] ...
[Listening] ...
[Listening] ...
[STT 255 ms] 'Got on how to memory.'

>>> Got on how to memory.
[Cathey] Sorry, I didn't catch your choice. Please answer again.
  type    : invalid
  reply   : Sorry, I didn't catch your choice. Please answer again.
  latency : 2215 ms  (llm: 2205 ms)
  reason  : clarification_not_resolved
[Listening] ...
[STT 245 ms] "Can't tell me a joke."

>>> Can't tell me a joke.
[Cathey] Let's start over. What would you like me to do?
  type    : invalid
  reply   : Let's start over. What would you like me to do?
  latency : 984 ms  (llm: 978 ms)
  reason  : clarification_max_retries
[Listening] ...
[STT 310 ms] 'This is all I know as well.'

>>> This is all I know as well.
  type    : invalid
  reply   : None
  latency : 0 ms  (llm: 0 ms)
  reason  : assistant_name_not_detected
[Listening] ...
[STT 1065 ms] "I'm going to show you what I'm going to do."

>>> I'm going to show you wha